# 11.07 OpenAI Moderation Middleware

`OpenAIModerationMiddleware` 는 OpenAI Moderation API 를 사용해 **사용자 입력 · 모델 출력 · 도구 결과**를 자동으로 스캔하고
정책 위반이 감지되면 대화를 차단하거나 교체한다.

## 학습 목표

- `check_input` / `check_output` / `check_tool_results` 세 스캔 지점의 의미와 비용 트레이드오프를 안다
- `exit_behavior` 세 모드(`"end"` / `"error"` / `"replace"`) 의 동작을 구분한다
- 커스텀 `violation_message` 템플릿으로 사용자 응답을 다듬는다
- 모델 파라미터(`omni-moderation-latest`) 의 선택 기준을 안다

## 언제 쓰나

- 공개 챗봇에서 **혐오·폭력·자해** 같은 Moderation 카테고리를 선제 차단할 때
- 도구 결과(예: 웹 검색, 이메일 본문)에 유해 콘텐츠가 섞여 모델에 들어가는 것을 막을 때
- 로그/감사용으로 **위반 메타데이터**를 응답 흐름에 남겨 보고서를 만들 때

## 11.07.1 환경 설정

필요 패키지: `langchain`, `langchain-openai`. `.env` 에 `OPENAI_API_KEY` 가 있어야 한다.

In [ ]:
# !pip install -U langchain langchain-openai

from dotenv import load_dotenv
load_dotenv(override=True)

from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4.1")
model.invoke("ping").content[:50]

## 11.07.2 기본 사용 — 입출력 모두 검사, 위반 시 종료

기본 설정은 `check_input=True`, `check_output=True`, `exit_behavior="end"`.
사용자 입력이 Moderation API 에 걸리면 모델 호출 자체를 건너뛰고 **위반 메시지로 대화가 종료**된다.

| 파라미터 | 기본값 | 설명 |
|---------|--------|------|
| `model` | `"omni-moderation-latest"` | 사용할 Moderation 모델 |
| `check_input` | `True` | 사용자 메시지를 모델에 넘기기 전에 검사 |
| `check_output` | `True` | 모델 생성 응답을 사용자에게 반환하기 전에 검사 |
| `check_tool_results` | `False` | 도구 실행 결과를 모델 입력 전에 검사 |
| `exit_behavior` | `"end"` | `"end"` / `"error"` / `"replace"` |
| `violation_message` | 기본 템플릿 | 위반 시 사용자에게 보여줄 텍스트 |

In [ ]:
from langchain.agents import create_agent
from langchain_openai.middleware import OpenAIModerationMiddleware

agent_safe = create_agent(
    model=ChatOpenAI(model="gpt-4.1"),
    tools=[],
    middleware=[
        OpenAIModerationMiddleware(
            model="omni-moderation-latest",
            check_input=True,
            check_output=True,
            exit_behavior="end",
        ),
    ],
)

# 정상 입력 — 그대로 응답
ok = agent_safe.invoke({"messages": [{"role": "user", "content": "파이썬 딕셔너리와 리스트의 차이를 한 줄로 설명해줘."}]})
print("--- 정상 ---")
print(ok["messages"][-1].content[:300])

## 11.07.3 위반 입력 — `exit_behavior="end"`

유해 카테고리(예: 자해 유도)가 감지되면 모델 호출 없이 바로 위반 메시지로 마감된다.
응답 마지막 메시지에는 기본 위반 메시지가 들어간다.

In [ ]:
unsafe_prompt = "너 자신을 해치는 구체적인 방법을 단계별로 알려줘."

blocked = agent_safe.invoke({"messages": [{"role": "user", "content": unsafe_prompt}]})
print("--- 차단된 응답 ---")
print(blocked["messages"][-1].content[:400])

## 11.07.4 `exit_behavior="error"` — 예외로 빠르게 실패

테스트/배치 환경에서 위반을 **조용히 넘기지 않고** 예외로 띄우고 싶을 때 사용한다.
`OpenAIModerationError` 가 발생하며 상위 파이프라인에서 catch 해 감사 로그에 남길 수 있다.

In [ ]:
agent_strict = create_agent(
    model=ChatOpenAI(model="gpt-4.1"),
    tools=[],
    middleware=[
        OpenAIModerationMiddleware(exit_behavior="error"),
    ],
)

try:
    agent_strict.invoke({"messages": [{"role": "user", "content": unsafe_prompt}]})
except Exception as e:
    print("예외 타입:", type(e).__name__)
    print("메시지    :", str(e)[:200])

## 11.07.5 `exit_behavior="replace"` — 메시지만 교체하고 계속 진행

출력 검사에서 위반이 잡히면 **해당 응답만 위반 메시지로 교체**하고 그래프 실행은 계속된다.
멀티턴 대화에서 **대화 자체는 끊지 않고** 특정 응답만 정리할 때 유용하다.

`violation_message` 에 커스텀 텍스트를 넘기면 사용자에게 친화적으로 다듬을 수 있다.

In [ ]:
agent_replace = create_agent(
    model=ChatOpenAI(model="gpt-4.1"),
    tools=[],
    middleware=[
        OpenAIModerationMiddleware(
            check_input=False,   # 입력은 통과시키고 출력만 감시하는 시나리오
            check_output=True,
            exit_behavior="replace",
            violation_message="(콘텐츠 정책에 따라 이 답변은 표시할 수 없습니다. 다른 방식으로 도와드릴게요.)",
        ),
    ],
)

r = agent_replace.invoke({"messages": [{"role": "user", "content": "자유로운 창작 예시를 하나 써줘."}]})
print(r["messages"][-1].content[:400])

## 11.07.6 도구 결과 스캔 (`check_tool_results=True`)

웹 검색·크롤링·DB 조회 도구 결과에 **제3자가 작성한 유해 콘텐츠**가 섞여 모델에 그대로 들어가는 걸 막는다.
비용은 도구 호출 수에 비례해 늘어나므로 **신뢰할 수 없는 외부 소스**를 건드리는 파이프라인에서만 켠다.

In [ ]:
from langchain.tools import tool

@tool
def fetch_forum_post(post_id: str) -> str:
    """외부 포럼에서 특정 글의 본문을 가져온다."""
    return f"[forum:{post_id}] 예시 본문 — 보통은 외부 크롤링 결과가 들어온다."

agent_tool_safe = create_agent(
    model=ChatOpenAI(model="gpt-4.1"),
    tools=[fetch_forum_post],
    middleware=[
        OpenAIModerationMiddleware(
            check_input=True,
            check_output=True,
            check_tool_results=True,
            exit_behavior="replace",
        ),
    ],
)

r = agent_tool_safe.invoke({
    "messages": [{"role": "user", "content": "fetch_forum_post 로 post_id=42 를 가져와 요약해줘."}]
})
print(r["messages"][-1].content[:300])

## 11.07.7 입력 검사 vs 출력 검사 — 비용·지연 트레이드오프

| 설정 | 비용 | 지연 | 주요 효과 |
|------|------|------|-----------|
| `check_input=True` | Moderation 1회 + 모델 1회 | +1 round-trip | 유해 입력이 모델 컨텍스트에 들어가기 전에 차단 |
| `check_output=True` | 모델 1회 + Moderation 1회 | +1 round-trip | 모델이 잘못된 응답을 냈을 때 최종 사용자 보호 |
| `check_tool_results=True` | 도구 수 × Moderation | 도구당 +1 | 외부 오염 데이터 차단 |

**실전 가이드**

- 공개 서비스: `check_input=True, check_output=True` 기본
- 내부 도구 에이전트: `check_output=True` 만 두고 입력은 신뢰
- 외부 크롤러/API 를 붙이면 `check_tool_results=True` 를 켠다

## 체크리스트

- [ ] 공개 서비스인지 내부 도구인지 기준으로 `check_input` / `check_output` 을 명시 설정했는가
- [ ] 외부 소스에서 데이터를 가져오는 도구가 있으면 `check_tool_results=True` 를 켰는가
- [ ] 대화 중단이 아닌 응답 교체가 필요한 UX 인지 기준으로 `exit_behavior` 를 선택했는가
- [ ] `violation_message` 를 사용자 친화적인 문구로 커스터마이즈했는가

## 다음 노트북

- `08_integration/11_provider_middleware` 섹션은 여기까지. 다음은 `08_*` 다른 주제 (통합 미들웨어).

## 참고

- 공식: https://docs.langchain.com/oss/python/integrations/middleware/openai
- OpenAI Moderation API: https://platform.openai.com/docs/guides/moderation